# Process GESTALT barcodes from serially sampled sperm data 
**Authors:** <br>
Step A: Connor Shrader and Andy Sposato <br>
Step B: Connor Shrader and Andy Sposato <br>
Step C: Drew Shapiro and Connor Shrader <br>
Step D: Connor Shrader <br>
Step E: Connor Shrader and Andy Sposato <br>
Step F: Connor Shrader, Andy Sposato & Kimberly Truong <br>

### A) Throw out whole files that have low total reads.
Each file contains the barcode reads for a single fish, sample, and replicate. <br>
If the number of barcode reads in the file is less than a given threshold, the data is to unreliable to be used and is thrown out.

Data that pass this step are saved to `all_data.csv`.

### B) Filter individual barcodes by reads and proportion cutoffs. 
The data contains many barcodes that have very few reads. To reduce this noise, we define a threshold number of reads that a barcode must appear in a replicate to be kept. Each barcode must also exceed some specified *proportion* of the total reads in the replicate.

Data that pass this step are saved to `filtered_data.csv`.

### C) Cluster barcodes that are nearly identical
The data contain many barcodes that come from nearly identical DNA sequences. To distinguish barcode that only differ by misreads from truly distinct barcodes, we cluster the sequences in each fish. We measure "distance" between sequences using the Levenshtein distance metric and generate a network that connects nearby sequences. Clusters are chosen to be the connected components of this network.

Data after this step are saved to `collapsed_data.csv`.

### D) Additional filtering steps

Some additional filtering steps are run to remove barcodes that only appeared in one sample, or that never appear in all three replicates.

Data after this step are saved to `refined_data.csv`.

### E) Examine questionable barcodes and identify contaminant barcodes. 

GESTALT editing should produce unique barcodes within each fish. However, in the data, we sometimes see reads from the same barcode in multiple fish. This step identifies the fish to which the barcode truly belongs, and removes that barcode from other fish. When a source fish cannot be identified, the barcode is removed from all fish.

Data after this step are saved to `classified_data.csv`.

### F) Make barcode matrix. 
A composite barcode matrix file containing barcode information for all fish in the dataset is written to the experiments folder and used for generating Muller plots. Barcode matrix files for each individual fish are also written to the experiments folder.

## Warnings: 
This python notebook file should be stored at the same level as `data/`. <br>

This pipeline will break if you change your naming system for sample replicates or mess with the data structure Kimberly established. See github.com/Gagnon-lab/Sperm-Analysis for Kimberly's pipeline.

### Settings

Step A:
* `total_read_cutoff`: ignore any `.allReadCounts` files with total reads strictly less than this value.
* `start_age`: the age that fish start getting sampled. This is used to create entries in `passed_samples_info.csv`.

Step B:
* `reads_cutoff`: remove a barcode from a sample if it has strictly less than this many reads.
* `prop_cutoff`: remove a barcode from a sample if it represents less than this proportion of the total reads.

Step C:
* `unedited`: The barcode of the unedited barcode.
* `cluster_barcodes`: If `False`, this step is not run.
* `edge_distance`: The maximum distance between two sequences to consider them connected. A larger value of `edge_distance` means more clustering.

Step D:
* `run_complete_triple_rep_filter`: remove a barcode in all fish if it never appears in all 3 replicates for any choice of fish/sample.
* `run_fish_triple_rep_filter`: remove a barcode from a fish if it never appears in all 3 replicates in any sample (if set to `True`, then the value of `run_complete_triple_rep_filter` does not matter)
* `fish_one_sample_filter`: remove a barcode from a fish if it only appears in one time sample.

Step E:
* `ratio_cutoff`: Any barcodes with a ratio score larger than this value are immediately assigned to the "winner" fish. Set this value to `False` to disable the ratio filter.
* `list_questionable_barcodes_before_classify`: Output a text file listing questionable barcodes before the classifying step.
* `list_questionable_barcodes_after_classify`: Output a text file listing barcodes that are *still* questionable after the classifying step.
* `image_questionable_barcodes_before_classify`: Plot a heatmap of all questionable barcodes before the classifying step.
* `image_questionable_barcodes_after_classify`: Plot a heatmap of all barcodes that are *still* questionable after the classifying step.
* `manual_classification`: a dictionary mapping barcodes to assigned outcomes. Ideally, this should be used only for debugging and testing. For example, setting `manual_classification = {'NONE_ABC_NONE': 'fish9'}` will assign fish9 as the "winner" fish. One can also set the value `'contaminant'` to remove the barcode.
* `delete_still_questionable_barcodes`: Whether to delete any barcodes that are still questionable after classification.

Step (F):
* `edited_fish_only`: If `False`, barcode matrices of control fish will be created, too.
* `fillna`: Whether to fill in empty values with 0's in the output barcode matrices.

In [1]:
# Step (A)
dataset = 'full-barcoded-fish'
total_read_cutoff = 1000
start_age = 4

# Step (B)
reads_cutoff = 10
prop_cutoff = 0.005

# Step (C)
unedited = "NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE"
cluster_barcodes = True
edge_distance = 2

# Step (D)
run_complete_triple_rep_filter = True
run_fish_triple_rep_filter = True
run_fish_one_sample_filter = True

# Step (E)
ratio_cutoff = 50

list_questionable_barcodes_before_classify = True
list_questionable_barcodes_after_classify = True
image_questionable_barcodes_before_classify = True
image_questionable_barcodes_after_classify = True

run_manual_classification = True
manual_classification = {}

delete_still_questionable_barcodes = True

# Step (F)
edited_fish_only = False
fillna = False

### Import packages, set up directory, and define helper functions

In [2]:
import datetime
import os 
import re
import operator
import collections
import numpy as np
import pandas as pd
from natsort import natsorted 
import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import wrap
import Levenshtein
import networkx as nx

import pipeline_utility as pu
import BarcodeRemap as br

Make sure that we are in the current working directory (`sperm-data-analysis/`)

In [3]:
cwd = os.getcwd()
if cwd.split('/')[-1] != 'sperm-data-analysis':
    raise Exception(f'Make sure you are running Python in the process_sperm_barcodes ' +
                    f'directory. You are currently running in {cwd}.')
else: print(f'We are in the current working directory: {cwd}/.')

We are in the current working directory: /Users/connor/Documents/github/sperm-data-analysis/.


Create a unique experiments subfolder to contain all the outputs from running the pipeline.

In [4]:
date = datetime.datetime.today().strftime('%m.%d.%Y') + "_" + \
       datetime.datetime.today().strftime('%I.%M_%p')

experiments_subfolder = "./experiments/" + date + "/"
                            
# if the experiments_subfolder doesn't already exist, create it
if not os.path.exists('./experiments'):
    os.mkdir('./experiments')
if not os.path.exists(experiments_subfolder): 
    os.mkdir(experiments_subfolder)

print(f'All outputs will be sent to {experiments_subfolder}.')

All outputs will be sent to ./experiments/09.13.2025_11.02_AM/.


Print the settings used to a text file titled `pipeline_settings.txt`.

In [5]:
summary = open(experiments_subfolder + "pipeline_settings.txt", "w")

def write_setting_line(file, var, name, buff):
    """
    Write a line of setting information to a file.

    'file': the file object we are writing to.
    'var':  the variable whose value we are storing.
    'name': the name of the variable to be written in the file.
    'buff': the buffer between the variable name and its value;
            used to evenly space information.
    
    """
    file.write(f"\t{(name + ' ').ljust(buff, '.')} {var}\n")
        
summary.write(f"Settings used during the pipeline run [{date}].\n\n")

summary.write("Step (A): throw out files with low read counts.\n")

buff = 21
write_setting_line(summary, dataset, 'dataset', buff)
write_setting_line(summary, total_read_cutoff, 'total_read_cutoff', buff)
write_setting_line(summary, start_age, 'start_age', buff)

summary.write("\nStep (B): filter barcodes.\n")
write_setting_line(summary, reads_cutoff, 'reads_cutoff', buff)
write_setting_line(summary, prop_cutoff, 'prop_cutoff', buff)

buff = 34
summary.write("\nStep (C): cluster barcodes.\n")
write_setting_line(summary, unedited, 'unedited', buff)
write_setting_line(summary, cluster_barcodes, 'cluster_barcodes', buff)
write_setting_line(summary, edge_distance, 'edge_distance', buff)
summary.write('\n')

buff = 34
summary.write("\nStep (D): additional filtering.\n")
write_setting_line(summary, run_complete_triple_rep_filter, 'run_complete_triple_rep_filter', buff)
write_setting_line(summary, run_fish_triple_rep_filter, 'run_fish_triple_rep_filter', buff)
write_setting_line(summary, run_fish_one_sample_filter, 'run_fish_one_sample_filter', buff)
summary.write('\n')

buff = 34
summary.write("\nStep (E): identify contaminant barcodes.\n")
write_setting_line(summary, ratio_cutoff, 'ratio_cutoff', buff)
summary.write('\n')

buff = 50
write_setting_line(summary, list_questionable_barcodes_before_classify, 'list_questionable_barcodes_before_classify', buff)
write_setting_line(summary, list_questionable_barcodes_after_classify, 'list_questionable_barcodes_after_classify', buff)
write_setting_line(summary, image_questionable_barcodes_before_classify, 'image_questionable_barcodes_before_classify', buff)
write_setting_line(summary, image_questionable_barcodes_after_classify, 'image_questionable_barcodes_after_classify', buff)

buff = 34
summary.write('\n')
write_setting_line(summary, run_manual_classification, 'run_manual_classification', buff)
summary.write('\n')
if run_manual_classification:
    summary.write(f"\tManual classifications were as follows:\n")
    for key, value in manual_classification.items():
        buff = 4 + max([len(key) for key in manual_classification.keys()])
        # summary.write(f"\t{key}: {value}\n")
        write_setting_line(summary, value, key, buff)
    summary.write("\n")

buff = 38
write_setting_line(summary, delete_still_questionable_barcodes, 'delete_still_questionable_barcodes', buff)

buff = 20
summary.write("\nStep (F): create barcode matrices.\n")
write_setting_line(summary, edited_fish_only, 'edited_fish_only', buff)
write_setting_line(summary, fillna, 'fillna', buff)
summary.close() 

## A) Throw out whole files that have low total reads and write passed-samples-info.csv.

Currently, Kimberly's code relies on the user writing a samples-info.csv from scratch. We want to filter samples that are totally garbage (all barcodes have small numbers of reads). Then write a samples-info.csv based on the .allReadCounts files that remain after that step. We'll have to rewrite .allReadCounts we want to keep with a "pass" key in the name so that we can direct next steps to read just those files. Next steps include removing barcodes with low read numbers and proportions and removing contaminant barcodes from their samples. 

#### Load all the .allReadCounts files in the data directory to `all_data`.

In [6]:
# Get list of file paths leading to .allReadCounts files
# At the same time, save the corresponding path "tuples" to a list.
# This will be used later to detect "empty" .allReadCounts files
# and output the correct values to the step A summary file.

filepaths = []
for dirpath, dirnames, filenames in os.walk(f"data/{dataset}/"):
   for filename in filenames:
      if filename.endswith(".allReadCounts"):
          filepath = os.path.join(dirpath, filename)
          filepaths.append(filepath)

filepaths = natsorted(filepaths)
all_file_tuples = [pu.tuple_from_file(fp) for fp in filepaths]
print(f"An example .allReadCounts filepath in your directory:\n{filepaths[0]}.")
print(f"An example file tuple: {all_file_tuples[0]}.")
print(f"\nIn total, {len(filepaths)} .allReadCounts files were found.")

An example .allReadCounts filepath in your directory:
data/full-barcoded-fish/fish1_s1_rep1/fish1_s1_rep1.allReadCounts.
An example file tuple: ('fish1', '1', '1').

In total, 1272 .allReadCounts files were found.


In [7]:
# Load all of the reads from .allReadCounts files to a pandas DataFrame.

# df_list is a list that will contain one data frame per file.
df_list = []

# Each file tuple (such as ('fish1', '1', '1')) will be assigned to either kept_groups
# (if the read count is sufficiently large), removed_groups (if it is not sufficiently large),
# or empty_groups (if the file is empty). We also track the number of reads per group, which
# is output in the Step A summary report.
kept_groups = []
removed_groups = []
empty_groups = []
reads_per_group = {}

for fish, sample, replicate in all_file_tuples:
    filename = f"{fish}_s{sample}_rep{replicate}"
    read_file = f"./data/{dataset}/{filename}/{filename}.allReadCounts"

    try:
        df = pd.read_csv(read_file, sep='\t', index_col='array')
    except Exception as e:
        print(f"Error reading {read_file}: {e}")
        continue

    # Edit the data frame and append it to our list of data frames.
    df.rename(columns={'event': 'barcode', 'count': 'reads'}, inplace=True)
    df.index.name = None
    df.insert(1, 'fish', fish)
    df.insert(2, 'sample', sample)
    df.insert(3, 'replicate', replicate)
    df.insert(4, 'file', filename)

    reads = df['reads'].sum()
    reads_per_group[(fish, sample, replicate)] = reads

    # Determine whether the file passes our initial filter.
    passed = reads >= total_read_cutoff
    df.insert(7, 'pass', passed)

    if passed:
        kept_groups.append((fish, sample, replicate))
    else:
        removed_groups.append((fish, sample, replicate))

    if not df.empty:
        df_list.append(df)
    else:
        empty_groups.append((fish, sample, replicate))

In [8]:
print(f'Of the {len(filepaths)} total files, {len(kept_groups)} passed ' + 
      f'the initial filter (they had over {total_read_cutoff} reads).')
      
print(f'Of the {len(removed_groups)} files that did not pass, {len(empty_groups)} {"was" if len(empty_groups) == 1 else "were"} completely empty.')

Of the 1272 total files, 1158 passed the initial filter (they had over 1000 reads).
Of the 114 files that did not pass, 0 were completely empty.


In [9]:
# Merge the list of data frames into a single data frame.
all_data = pd.concat(df_list, ignore_index = True)
print(f'The number of entries in all_data is {len(all_data)}. The first five rows are below:')
all_data.head()

The number of entries in all_data is 1023042. The first five rows are below:


,barcode,fish,sample,replicate,file,reads,proportion,pass
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530,True
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True


In [10]:
# Remove all files where used entries are less than 0.3
info_df = pd.DataFrame({'sample' : [], 'usedEntries' : [], 'totalEntries' : [], 'dir' : []})
for dire in os.listdir('data/mergedinfos'):
    if 'OUTPUT' in dire:
        path = f'data/mergedinfos/{dire}/html'
        inners = os.listdir(f'{path}')
        if '.DS_Store' in inners:
            inners.remove('.DS_Store')
        new_df = pd.read_csv(f'{path}/{inners[0]}/merged.info', sep='\t')
        new_df['dir'] = dire
        info_df = pd.concat([info_df, new_df])
info_df['file'] = info_df['sample']
info_df['usedEntryProp'] = info_df['usedEntries'] / info_df['totalEntries']

keep = []
for df in df_list:
    file = df['file'].iloc[0]
    try:
        used_prop = info_df[info_df['file'] == file]['usedEntryProp'].iloc[0]
        passed = used_prop > 0.3
        if passed:
            keep.append(file)
    except Exception as e:
        print('merged.info not found for file ' + file)

merged.info not found for file fishwater_s4_rep2


In [11]:
# Throw away all reads from files that didn't pass used entries test
all_data = all_data[all_data['file'].isin(keep)].copy()
print(f'The number of entries in all_data is {len(all_data)}. The first five rows are below:')
all_data.head()

The number of entries in all_data is 1008924. The first five rows are below:


,barcode,fish,sample,replicate,file,reads,proportion,pass
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530,True
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True


In [12]:
# Throw away all sample 3's!
all_data = all_data[all_data['sample'] != '3'].copy()
len(all_data)

914852

In [13]:
# Save all_data to a file.
all_data.to_csv(f'{experiments_subfolder}all_data.csv', index = False)

In [14]:
# Write a `passed-samples-info.csv file`.
sample_info = []
for fish, sample, replicate in kept_groups:
    sample_id = f'{fish}_s{sample}_rep{replicate}'
    sample_info.append({'sample_id': sample_id, 'fish': fish[4:], 
                        'age': int(sample) + start_age, 'replicate': replicate, 
                        'condition': 'control' if 'ctrl' in fish else 'edited'})
    
passed_samples_info = pd.DataFrame(sample_info)
passed_samples_info.to_csv(experiments_subfolder + 'passed_samples_info.csv', index = False)
passed_samples_info.head()

,sample_id,fish,age,replicate,condition
0,fish1_s1_rep1,1,5,1,edited
1,fish1_s1_rep2,1,5,2,edited
2,fish1_s1_rep3,1,5,3,edited
3,fish1_s2_rep1,1,6,1,edited
4,fish1_s2_rep2,1,6,2,edited


#### Step A summary report.

In [15]:
all_groups = kept_groups + removed_groups

summary = open(experiments_subfolder + "stepA_summary.txt", "w")
summary.write("Summary of step A: Throw out whole files that have low total reads and write passed-samples-info.csv.\n\n")
summary.write("Your data contains " + str(len(all_groups)) + ' .allReadCounts files, or ' + str(len(all_groups)) + ' samples including replicates.\n\n')
summary.write("If a file contained at least " + str(total_read_cutoff) + " total reads across all barcodes, then it has passed this filter step.\n") 
summary.write("\n")
if len(removed_groups) == 0: 
    summary.write("All of your files passed this step.\n")
elif len(removed_groups) > 0:
    summary.write(f'{len(removed_groups)} files failed this step:\n')

    for group in removed_groups: 
        fish, sample, replicate = group
        sample_id = f'{fish}_s{sample}_rep{replicate}'
        summary.write(f'{sample_id} failed with only {reads_per_group[group]} total reads. \n')
            
summary.write("\n")
summary.write(f'{len(kept_groups)} files passed the filter.\n\n')

# summary.write("The contents of read counts from all files was written in a new file titled 'all_data.csv.\n")
# summary.write("The contents of read counts that passed were copied to a new file titled 'pass_data.csv' \nin the same subfolder.\n\n")
# summary.write("The passed-samples-info.csv was also written and saved to the folder called data.\nIt contains only samples that pass this step.\n")

summary.write("""\
    The contents of read counts from all files was written in a new file titled 'all_data.csv.
    The contents of read counts that passed were copied to a new file titled 'pass_data.csv'
    in the same subfolder.

    The passed-samples-info.csv was also written and saved to the folder called data.
    It contains only samples that pass this step.
""")
summary.close() 

## B) Filter barcodes by reads and proportion cutoffs.

#### Apply the filters and summarize results

In [16]:
pass_data = all_data[all_data['pass']].drop(columns='pass').copy()

pass_data['reads_filter'] = pass_data['reads'] >= reads_cutoff
pass_data['prop_filter'] = pass_data['proportion'] >= prop_cutoff

In [17]:
pass_data.to_csv(experiments_subfolder + 'pass_data.csv', index = False)

In [18]:
filtered_data = pass_data[pass_data['reads_filter'] & pass_data['prop_filter']] \
                .drop(columns=['reads_filter', 'prop_filter']).copy()

In [19]:
# Track how many reads/barcodes were removed for each sample
pass_aggregate = pass_data.groupby(by = ['fish', 'sample', 'replicate'])['reads'] \
                          .agg(['size', 'sum']) \
                          .rename(columns = {'size': 'pass_size', 'sum': 'pass_sum'})

filtered_aggregate = filtered_data.groupby(by = ['fish', 'sample', 'replicate'])['reads'] \
                                  .agg(['size', 'sum']) \
                                  .rename(columns = {'size': 'filtered_size', 'sum': 'filtered_sum'})

step_b_summary = pd.concat([pass_aggregate, filtered_aggregate], axis = 1)
step_b_summary.head()

pass_size  pass_sum  filtered_size  filtered_sum
fish  sample replicate                                                  
fish1 1      1                710     76693              6         72943
             2                451     40635              8         38979
             3                630     52249              6         49230
      10     1                448     40540             12         37813
             2                464     41842             11         38952

#### Step B summary report

In [20]:
summary = open(experiments_subfolder + "stepB_summary.txt", 'w')
summary.write("Summary of step B: Filter barcodes by reads and proportion cutoffs.\n\n")
summary.write("For each .passReadCounts file, if a barcode had fewer than " + str(reads_cutoff) + " reads or a proportion below " + str(prop_cutoff) + ",\nit was removed from the file.\n\n")
summary.write("A list of barcodes that were removed was written to a sample file that ends with 'stepB_removed_barcodes.txt'\nThis may be useful to verify that the changes from .passReadCounts to .filteredReadCounts were accurate.\n\n")
summary.write("Here is a recap of the filtering for each sample: \n\n") 
summary.write("**************************************************************************************** \n\n")

filepath_string = './SpermAnalysis_Muller1/data/viz-Output/' 

for group in step_b_summary.index: 
    fish, sample, replicate = group
    sample_id = f'{fish}_s{sample}_rep{replicate}'

    pass_read_counts = step_b_summary.loc[group, "pass_sum"]
    pass_count = step_b_summary.loc[group, "pass_size"]
    filtered_read_counts = step_b_summary.loc[group, "filtered_sum"]
    filtered_count = step_b_summary.loc[group, "filtered_size"]
    
    reads_removed = pass_read_counts - filtered_read_counts
    summary.write(sample_id + " had " + str(pass_read_counts) + " reads in the .passReadCounts file. Filtering removed " + str(reads_removed) + ' reads.\n')
    summary.write("There are now " + str(filtered_read_counts) + " reads in the " + sample_id + ".filteredReadCounts file.\n\n")
    
    barcodes_removed = pass_count-filtered_count
    summary.write(sample_id + " had " + str(pass_count) + " barcodes in the .passReadCounts file. Filtering removed " + str(barcodes_removed) + ' barcodes.\n')
    if filtered_count == 1: 
        summary.write("There is now " + str(filtered_count) + " barcode in the " + sample_id + ".filteredReadCounts file.\n\n")
    else:     
        summary.write("There are now " + str(filtered_count) + " barcodes in the " + sample_id + ".filteredReadCounts file.\n\n")
    summary.write("**************************************************************************************** \n\n")
summary.close()

## C) Barcode Clustering

In [21]:
collapsed_data = filtered_data.copy()

In [22]:
if cluster_barcodes:
    reference_seq = br.BarcodeRemap().zebrafish_reference_sequence
    collapsed_data['sequence'] = collapsed_data['barcode'].apply(lambda x: br.get_sequence(x, reference_seq) \
                                                          .upper())

    # grouped_data combines the reads of each barcode from different samples and replicates.
    grouped_data = collapsed_data.groupby(['fish', 'sequence', 'barcode'], as_index = False) \
                                ['reads'].sum() \
                                .sort_values('reads', ascending = False) \
                                .reset_index(drop=True)
    
    directory = f'{experiments_subfolder}clustering'
    if not os.path.exists(directory): 
        os.mkdir(directory)

In [23]:
import importlib
importlib.reload(pu)

<module 'pipeline_utility' from '/Users/connor/Documents/github/sperm-data-analysis/pipeline_utility.py'>

In [24]:
if cluster_barcodes:
    for fish in all_data['fish'].unique():
        file = open(experiments_subfolder + f"clustering/{fish}.txt", 'w')
        
        fish_data = grouped_data.query('fish == @fish') \
                                .reset_index(drop=True)
        sequences = np.array(fish_data['sequence'])
    
        file.write(f"There are {len(sequences)} unique sequences in {fish}.\n")
    
        distances = pu.compute_levenshtein_matrix(sequences)
        components = pu.find_connected_components(distances, threshold = edge_distance)

        # Each component corresponds to a cluster.
        for i, component in enumerate(components):
            comp_barcodes = fish_data['barcode'].iloc[list(component)]
            comp_data = fish_data[fish_data['barcode'].isin(comp_barcodes)]
            
            pu.summarize_component(i, comp_data, distances, file)

            # Identify which barcode is the 'true' barcode. All other
            # barcodes in this cluster are mapped to this one.
            winner = pu.resolve_winner_barcode(comp_data, unedited)
    
            collapsed_data.loc[
                (collapsed_data['fish'] == fish) & 
                (collapsed_data['barcode'].isin(comp_barcodes)), 'barcode'
            ] = winner

            filtered_data.loc[
                (filtered_data['fish'] == fish) & 
                (filtered_data['barcode'].isin(comp_barcodes)), 'clustered_barcode'
            ] = winner
            
            file.write('\n')
    
        summary.close() 

In [25]:
if cluster_barcodes:
    # Combine rows that share the same (reassigned) barcode
    collapsed_data = (
        collapsed_data.groupby(['barcode', 'fish', 'sample', 'replicate', 'file'], as_index=False)
                      .agg({'reads': 'sum', 'proportion': 'sum'})
                      .sort_values(['fish', 'sample', 'replicate'])
    )

In [26]:
filtered_data.to_csv(f'{experiments_subfolder}filtered_data.csv', index = False)

In [27]:
collapsed_data.drop(columns = 'sequence', inplace = True, errors = 'ignore')

## D) More filters

In [28]:
new_columns = ['complete_triple_rep_classification', 'fish_triple_rep_classification',
               'fish_one_sample_classification']

for new_column in new_columns:
    collapsed_data.loc[:, new_column] = True
    collapsed_data[new_column] = collapsed_data[new_column].astype('bool')

Determine values of `complete_triple_rep_classification`, `fish_triple_rep_classification`, and `fish_one_sample_classification`.

In [29]:
for barcode in collapsed_data['barcode'].unique():
    single_barcode = collapsed_data.query('barcode == @barcode')
    
    # grouped is a DataFrame with columns (fish, sample, replicate). The value of 'replicate' is
    # the count of replicates in which there is at least one read for that fish and sample.
    grouped = single_barcode.groupby(['fish', 'sample'], observed=True, as_index=False) \
                            ['replicate'].count() \
                            .reset_index()

    if grouped.replicate.max() < 3 and run_complete_triple_rep_filter:
        collapsed_data.loc[
            collapsed_data['barcode'] == barcode, 
            'complete_triple_rep_classification'
        ] = False
        
    for fish in single_barcode['fish'].unique():
        # grouped is a DataFrame with columns (sample, replicate). The value of 'replicate' is the count
        # of replicates in which there is at least one read for that sample.grouped_fish = single_barcode.query("fish == @fish").groupby('sample')['replicate'].count().reset_index()
        grouped_fish = single_barcode.query('fish == @fish').groupby('sample') \
                                     ['replicate'].count() \
                                     .reset_index()

        if grouped_fish.replicate.max() < 3 and run_fish_triple_rep_filter:
            # never appears in 3 reps, throw out
            collapsed_data.loc[
                (collapsed_data['barcode'] == barcode) & (collapsed_data['fish'] == fish), 
                'fish_triple_rep_classification'
            ] = False
            
        if len(grouped_fish) == 1 and run_fish_one_sample_filter:
            # only appears in one sample, throw out
            collapsed_data.loc[
                (collapsed_data['barcode'] == barcode) & (collapsed_data['fish'] == fish), 
                'fish_one_sample_classification'
            ] = False

In [30]:
# refined_data contains all the rows of collapsed_data such that the three classification
# columns (defined above) are `True`.
refined_data = collapsed_data[collapsed_data.filter(like='_classification').all(True)].copy()

In [31]:
collapsed_data.to_csv(f'{experiments_subfolder}collapsed_data.csv', index = False)

In [32]:
print(f'''\
Before initial classification, there were {len(collapsed_data)} rows.
After classification, there are now {len(refined_data)} rows.\
    ''')

Before initial classification, there were 9172 rows.
After classification, there are now 7210 rows.    


## E) Examine questionable barcodes and identify contaminant barcodes. 

Questionable barcodes are barcodes that show up in multiple animals. They are likely contamination. We want to confidently remove them from samples where they do not belong. Before we can remove them from the appropriate files, we need to examine each according to the number of reads that support them in every fish. 

### Find questionable barcodes
These are barcodes that show up in multiple fish. 

A barcode will not be tagged questionable if it shows up in multiple samples or replicates of a single fish. 

In [33]:
questionable_barcodes = pu.identify_questionable(refined_data, unedited)
print(f'There are {len(questionable_barcodes)} questionable barcodes.')

There are 19 questionable barcodes.


In [34]:
refined_data.loc[:, 'manual_classification'] = True
refined_data.loc[:, 'ratio_classification'] = True

In [35]:
# Process any barcodes that have manual assignments set by the user.
for key, value in manual_classification.items():
    if value == 'contaminant':
        refined_data.query("barcode == @key").assign(manual_classification = False)
    else:
        refined_data.query('barcode == @key and fish != @value').assign(manual_classification = False)

In [36]:
questionable_after_manual = pu.identify_questionable(
    refined_data[refined_data['manual_classification']],
    unedited
)
print(f'Before manual classification, {len(questionable_barcodes)} barcodes were questionable. ' +
      f'{len(manual_classification)} had manual assignment. {len(questionable_after_manual)} ' +
      f'are still questionable.')

Before manual classification, 19 barcodes were questionable. 0 had manual assignment. 19 are still questionable.


In [37]:
if ratio_cutoff is not False:
    for barcode in questionable_after_manual:
        one_barcode = refined_data[refined_data.barcode == barcode]
        
        aggregate = one_barcode.groupby(by = 'fish', as_index = False, observed = True) \
                                      .aggregate({'reads': 'sum'})
        
        aggregate.sort_values('reads', ascending = False, inplace = True, ignore_index = True)
        
        winner_reads = aggregate.reads.iloc[0]
        winner_fish = aggregate.fish.iloc[0]
        
        total_reads = aggregate.reads.sum()
        
        ratio = winner_reads / (total_reads - winner_reads)

        if ratio > ratio_cutoff:
            refined_data.update(
                refined_data.query('barcode == @barcode and fish != @winner_fish') \
                            .assign(ratio_classification = False)
            )

In [38]:
refined_data.to_csv(experiments_subfolder + 'refined_data.csv', index = False)

In [39]:
classified_data = refined_data.query("manual_classification and ratio_classification").copy()

In [40]:
questionable_after_ratio = pu.identify_questionable(classified_data, unedited)
print(f'There are {len(questionable_after_ratio)} barcodes that are still questionable.')

There are 14 barcodes that are still questionable.


In [41]:
classified_data['no_longer_questionable'] = ~classified_data['barcode'].isin(questionable_after_ratio)
classified_data.drop(columns=['complete_triple_rep_classification', 'fish_triple_rep_classification',
                              'fish_one_sample_classification', 'manual_classification', 'ratio_classification'], inplace = True)
classified_data.head()

,barcode,fish,sample,replicate,file,reads,proportion,no_longer_questionable
347,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True
383,12D+34_12D+60_NONE_NONE_NONE_NONE_54D+199_54D+...,fish1,1,1,fish1_s1_rep1,643,0.008384,True
1534,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True
1548,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True
1615,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True


In [42]:
classified_data.to_csv(experiments_subfolder + 'classified_data.csv', index = False)

Remove any still questionable barcodes if `delete_still_questionable_barcodes` is True.

In [43]:
final_data = None
if delete_still_questionable_barcodes:
    final_data = classified_data[classified_data['no_longer_questionable']].copy()
else:
    final_data = classified_data

### Associate each barcode with a contamination status

In [44]:
questionable_barcodes_summary = dict()
for bc in questionable_barcodes:
    query = classified_data[classified_data.barcode == bc].fish.unique()
    
    if bc in manual_classification:
        questionable_barcodes_summary[bc] = f'{manual_classification[bc]} (manual classification)'
    elif bc in questionable_after_ratio:
        questionable_barcodes_summary[bc] = "still questionable"
    elif len(query) == 0:
        questionable_barcodes_summary[bc] = "contaminant"
    elif len(query) == 1:
        fish = query[0]
        questionable_barcodes_summary[bc] = f'{fish} (found by ratio)'
    else:
        raise Exception(f'barcode {bc} is weird.')

### Output findings

In [45]:
# write questionable barcodes to a file
if list_questionable_barcodes_before_classify:
    output = open(f'{experiments_subfolder}questionable_barcodes_list.txt', 'w')
    for barcode in questionable_barcodes: 
        output.write(f'{barcode}: {questionable_barcodes_summary[barcode]}')
        output.write("\n")
    output.close()

if list_questionable_barcodes_after_classify and len(questionable_after_ratio) > 0:
    output = open(f'{experiments_subfolder}still_questionable_barcodes_list.txt', 'w')
    for barcode in questionable_after_ratio: 
        output.write(barcode)
        output.write("\n")
    output.close()

In [46]:
# produce heatmaps
if image_questionable_barcodes_before_classify:
    directory = f'{experiments_subfolder}heatmaps_before_classification'
    if not os.path.exists(directory): 
        os.mkdir(directory)
    data = refined_data.copy()
    for i, bc in enumerate(questionable_barcodes):
        _, ratio = pu.summarize_barcode(data, bc)
        fig, ax = pu.heatmap(data, bc)
        ax.set_title("\n".join(wrap(f"{bc}", 50) + [f"\nratio: {ratio:.2f}, outcome: {questionable_barcodes_summary[bc]}"]))
        fig.tight_layout()
        plt.savefig(f"{directory}/barcode[{i}].png")
        plt.close()

if image_questionable_barcodes_after_classify:
    directory = f'{experiments_subfolder}heatmaps_after_classification'
    if not os.path.exists(directory): 
        os.mkdir(directory)
    data = classified_data.copy()
    for i, bc in enumerate(questionable_barcodes):
        if bc not in questionable_after_ratio: continue
        _, ratio = pu.summarize_barcode(data, bc)
        fig, ax = pu.heatmap(data, bc)
        ax.set_title("\n".join(wrap(f"{bc}", 50) + [f"\nratio: {ratio:.2f}, outcome: {questionable_barcodes_summary[bc]}"]))
        fig.tight_layout()
        plt.savefig(f"{directory}/barcode[{i}].png")
        plt.close()

In [47]:
classified_data.to_csv(experiments_subfolder + 'classified_data.csv', index = False)

In [48]:
len(refined_data)

7210

In [49]:
summary = open(experiments_subfolder+"stepE_summary.txt", 'w')
summary.write(
"""
Summary of step E: Remove contaminant barcodes from their read files.

The barcodes in questionable barcodes list were examined carefully
for their presence across animals. If a clear source of that barcode
could be found, the questionable barcode was assigned a source fish.
If there was not a clear winner, the questionable barcode was
considered contaminant everywhere.

The data before contaminant detection is stored in `refined_data.csv`.
The data after contaminant detection is stored in `classified_data.csv`.
Data with the contaminants removed is stored in `final_data.csv`.

Here is a recap of the filtering for each sample:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
""")

# for fish, sample, replicate in kept_groups:
#     sample_id = f'{fish}_s{sample}_rep{replicate}'
#     query = f'fish == @fish & sample == @sample & replicate == @replicate'

for fish in refined_data['fish'].unique():
    sample_id = fish
    query = f'fish == @fish'
    
    refined_slice = refined_data.query(query).copy()
    refined_barcodes = refined_slice.barcode.unique()
    
    classified_slice = classified_data.query(query).copy()
    classified_barcodes = classified_slice.barcode.unique()

    barcodes_removed = set(refined_barcodes) - set(classified_barcodes)
    reads_removed = refined_slice.reads.sum() - classified_slice.reads.sum()

    summary.write(
f"""
{sample_id} had {refined_slice.reads.sum()} reads and {len(refined_barcodes)}
before classification. Now, there are {classified_slice.reads.sum()} reads 
and {len(classified_barcodes)} after classification.

Barcode     | Reads before | Reads after
------------+--------------+------------
""")    
    for barcode in refined_barcodes:
        c_reads = classified_slice.query('barcode == @barcode')['reads'].sum()
        r_reads = refined_slice.query('barcode == @barcode')['reads'].sum()
        if c_reads != r_reads:
            summary.write(f'{barcode} | {r_reads} | {c_reads}\n')
        
    # if len(barcodes_removed) == 0: 
    #     summary.write("No barcodes were removed from " + sample_id + '.filteredReadCounts. \n')
    # else:
    #     if len(barcodes_removed) == 1:
    #         summary.write("This barcode was removed from " + sample_id + '.filteredReadCounts: \n')
    #     else:
    #         summary.write("These barcodes were removed from " + sample_id + '.filteredReadCounts: \n')
    #     for barcode in barcodes_removed:
    #         if barcode in questionable_barcodes_summary: 
    #             if questionable_barcodes_summary[barcode] == 'contaminant': 
    #                 summary.write('\t' + barcode + ": promiscuous barcode\n" )
    #             else: 
    #                 summary.write('\t' + barcode + ": contamination from " + questionable_barcodes_summary[barcode] + "\n")    
    summary.write("\n")
    summary.write("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ \n\n")

summary.write("\n")
summary.close()

### Fix proportions

In [50]:
# For each fish, sample, replicate, count the proportion of reads in the final
# dataset compared to the allreadycounts data.
prop_reads_kept = final_data.groupby(['fish', 'sample', 'replicate', 'file'], as_index = False)['proportion'].sum()

In [51]:
# ADDITIONAL FILTERING
# Filter out files with less than a proportion of 0.3 kept reads
prop_reads_kept['pass'] = prop_reads_kept['proportion'] > 0.3
print(prop_reads_kept['pass'].value_counts())

means = {}
for fish in prop_reads_kept['fish'].unique():
    filtered_prop_series = prop_reads_kept[(prop_reads_kept['fish'] == fish) & (prop_reads_kept['pass'])]['proportion']
    means[fish] = (filtered_prop_series.mean(), filtered_prop_series.std())

prop_reads_kept['mean'] = prop_reads_kept['fish'].apply(lambda fish: means[fish][0])
prop_reads_kept['std'] = prop_reads_kept['fish'].apply(lambda fish: means[fish][1])
prop_reads_kept['pass'] = prop_reads_kept['pass'] & ~(prop_reads_kept['proportion'] < prop_reads_kept['mean'] - 3 * prop_reads_kept['std'])
print(prop_reads_kept['pass'].value_counts())

passing_files = prop_reads_kept[prop_reads_kept['pass']]['file']
final_data = final_data[final_data['file'].isin(passing_files)]

prop_reads_kept.to_csv(experiments_subfolder + 'prop_reads_kept.csv', index = False)
prop_reads_kept.head()

pass
True     794
False    218
Name: count, dtype: int64
pass
True     777
False    235
Name: count, dtype: int64


,fish,sample,replicate,file,proportion,pass,mean,std
0,fish1,1,1,fish1_s1_rep1,0.951104,True,0.915882,0.054898
1,fish1,1,2,fish1_s1_rep2,0.959247,True,0.915882,0.054898
2,fish1,1,3,fish1_s1_rep3,0.942219,True,0.915882,0.054898
3,fish1,10,1,fish1_s10_rep1,0.932733,True,0.915882,0.054898
4,fish1,10,2,fish1_s10_rep2,0.930931,True,0.915882,0.054898


In [52]:
final_data.loc[:, 'proportion'] = (
    final_data['proportion'] /
    final_data.groupby(['fish', 'sample', 'replicate'])['proportion'].transform('sum')
)

In [53]:
final_data.to_csv(experiments_subfolder + 'final_data.csv', index = False)

## E) Make barcode matrix.

The barcode matrix will be used as input when generating Muller plots in R. 

In [54]:
print(os.getcwd())

/Users/connor/Documents/github/sperm-data-analysis


### Assemble the matrix

In [55]:
d = final_data.copy()
d['sample'] = pd.to_numeric(d['sample'])
d['sample_id'] = d['fish'] + '_s' + d['sample'].apply(str) + '_rep' + d['replicate']
# classified_data.sort_values('sample', inplace = True)

In [56]:
directory = experiments_subfolder + 'matrix'
if not os.path.exists(directory): 
    os.mkdir(directory)

fish_list = d.fish.unique()
for fish in fish_list:
    fish_df = d[d['fish'] == fish].copy()
    pt = fish_df.pivot_table(columns = ['sample_id'], index = 'barcode', values = 'reads', sort = False)
    if fillna:
        pt.fillna(0, inplace = True)
    pt.to_csv(f'{directory}/{fish}_barcodeMatrix_3filters.tsv', sep = '\t')

pt = d.pivot_table(columns = ['sample_id'], index = 'barcode', values = 'reads', sort = False)
if fillna:
    pt.fillna(0, inplace = True)
pt.to_csv(f'{directory}/barcodeMatrix_3filters.tsv', sep = '\t')

### Step E summary report

In [57]:
summary = open(experiments_subfolder+"stepE_summary.txt", 'w')
summary.write("Summary of step E: A barcode matrix file is written to the experiments folder and\nis used for generating Muller plots.\n\n")
summary.write("A barcode matrix file is also written for each fish in your samples.\n")
summary.write("The barcodes present in these matrix files will have passed all three filters (steps A, B and D)")
summary.write("\n")
summary.close()

# Job report

In [58]:
summary = open(experiments_subfolder+"full_job_report.txt", 'w')
stepA = open(experiments_subfolder+"stepA_summary.txt", 'r')
stepB = open(experiments_subfolder+"stepB_summary.txt", 'r')
# stepC = open(experiments_subfolder+"stepC_summary.txt", 'r')
stepD = open(experiments_subfolder+"stepD_summary.txt", 'r')
stepE = open(experiments_subfolder+"stepE_summary.txt", 'r')

summary.write("Processed GESTALT barcodes from serially sampled sperm data.\n")
summary.write("Your job completed at " + datetime.datetime.now().strftime('%I:%M %p on %m/%d/%Y'))
summary.write("\nAndy Sposato wrote original code for steps A-D.\nAndy adapted the code in Step E from original code written by Kimberly Truong.")
summary.write("\n\n############################################################################################\n\n")
for line in stepA.readlines(): 
    summary.write(line)
stepA.close()
summary.write("\n############################################################################################\n\n")
for line in stepB.readlines(): 
    summary.write(line)
stepB.close()
# summary.write("############################################################################################\n\n")
# for line in stepC.readlines(): 
#     summary.write(line)
# stepC.close()
summary.write("############################################################################################\n\n")
for line in stepD.readlines(): 
    summary.write(line)
stepD.close()
summary.write("############################################################################################\n\n")
for line in stepE.readlines(): 
    summary.write(line)
stepE.close()
summary.write("\n############################################################################################\n\n")
summary.write("It was a pleasure filtering your sperm barcodes. \n")
summary.close()

FileNotFoundError: [Errno 2] No such file or directory: './experiments/09.13.2025_11.02_AM/stepD_summary.txt'